In [ ]:
%pip install -q -e .. matplotlib

# 02 — FWI map (single day)

The Fire Weather Index (FWI) is part of the Canadian Forest Fire Weather Index System,
used operationally by EFFIS, Copernicus, and EUMETSAT.
This notebook fetches one day over Spain and plots it with the standard 6-class
danger scale.

## 1. Discover & open

In [ ]:
import httpx

server = "https://edr.example.com"  # replace with your EDR server root

collections = httpx.get(f"{server}/collections").raise_for_status().json()["collections"]
collection_id = collections[0]["id"]
collection_url = f"{server}/collections/{collection_id}"

In [ ]:
import xarray as xr

import edr_xarray  # registers engine="edr"

spain = (-9.5, 36.0, 3.3, 43.8)  # (lon_min, lat_min, lon_max, lat_max)
ds = xr.open_dataset(collection_url, engine="edr", bbox=spain)
ds

## 2. Pick a date

In [ ]:
print("from:", ds.t.values[0])
print("to:  ", ds.t.values[-1])

In [ ]:
date = "2024-08-01"  # CHANGE ME — any date in the range above

## 3. Plot with EFFIS classes

FWI uses a standard 6-class danger scale (Very low → Extreme).

In [ ]:
# EFFIS 6-class fire danger scale
levels = [0.0, 5.2, 11.2, 21.3, 38.0, 50.0]
colors = ["#008000", "#FFFF00", "#FFA500", "#FF0000", "#654321"]

var = next(iter(ds.data_vars))
ds[var].sel(t=date).plot(
    figsize=(10, 6),
    levels=levels,
    colors=colors,
    cbar_kwargs={"label": f"{var} (Very low → Extreme)"},
)